# Brain — Local CUDA Training Notebook
# =====================================
# Run this locally on any CUDA GPU (RTX 4050 / 3060 / etc.)
# Open a terminal in the brain/ project root and run:
#   jupyter notebook colab/train_brain.ipynb
#
# What gets trained:
#   1. RSSM World Model  (~4.5M params, ~20 min on RTX 4050)
#   2. TFT Forecaster    (~0.7M params, ~15 min on RTX 4050)
#   3. Actor-Critic      (~1.5M params, ~10 min on RTX 4050)
#
# Output: 3 .pt files saved to  data/models/
# Requirements: Python 3.10+, CUDA GPU, internet (for yfinance data)

## Section 1: Setup

In [1]:
# ── Cell 1: Verify GPU + install missing deps ────────────────────────────────
import subprocess, sys, os

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

required = ["torch", "pandas", "numpy", "pyarrow", "yfinance", "tqdm", "lightgbm"]
for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        pip_install(pkg)

# Fix CUDA DLL loading on Windows — add torch/lib to PATH
import importlib, site
for sp in site.getusersitepackages() if isinstance(site.getusersitepackages(), list) else [site.getusersitepackages()]:
    torch_lib = os.path.join(sp, "torch", "lib")
    if os.path.isdir(torch_lib) and torch_lib not in os.environ.get("PATH", ""):
        os.environ["PATH"] = torch_lib + os.pathsep + os.environ.get("PATH", "")
        print(f"Added to PATH: {torch_lib}")

import torch
assert torch.cuda.is_available(), (
    "CUDA not found!\n"
    "Make sure:\n"
    "  1. You have an NVIDIA GPU\n"
    "  2. CUDA drivers are installed: https://developer.nvidia.com/cuda-downloads\n"
    "  3. PyTorch CUDA build is installed:\n"
    "     pip install torch --index-url https://download.pytorch.org/whl/cu126"
)

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
print(f"GPU  : {props.name}")
print(f"VRAM : {vram_gb:.1f} GB")
print(f"PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}")

# RTX 4070 has 8 GB — pick batch sizes accordingly
VRAM_GB = vram_gb
WM_BATCH  = 4  if VRAM_GB < 7 else 8   if VRAM_GB < 12 else 16
TFT_BATCH = 16 if VRAM_GB < 7 else 32  if VRAM_GB < 12 else 64
print(f"\nBatch sizes → WorldModel: {WM_BATCH}, TFT: {TFT_BATCH}")

Added to PATH: C:\Users\M.S. BAVESH REDDY\AppData\Roaming\Python\Python313\site-packages\torch\lib
GPU  : NVIDIA GeForce RTX 4070 Laptop GPU
VRAM : 8.6 GB
PyTorch 2.10.0+cu126  |  CUDA 12.6

Batch sizes → WorldModel: 8, TFT: 32


In [2]:
# ── Cell 2: Set up local paths ────────────────────────────────────────────────
from pathlib import Path

# Hardcoded to the brain/ project root on this machine
BASE_DIR = Path(r'C:\Users\M.S. BAVESH REDDY\Desktop\brain')

# Create directories if missing
RAW_DIR  = BASE_DIR / "data" / "raw"
SAVE_DIR = BASE_DIR / "data" / "models"
RAW_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {BASE_DIR}")
print(f"Raw data     : {RAW_DIR}")
print(f"Models saved : {SAVE_DIR}")

Project root : C:\Users\M.S. BAVESH REDDY\Desktop\brain
Raw data     : C:\Users\M.S. BAVESH REDDY\Desktop\brain\data\raw
Models saved : C:\Users\M.S. BAVESH REDDY\Desktop\brain\data\models


In [3]:
# ── Cell 3: Imports + constants ───────────────────────────────────────────────
import math, time, gc, zipfile, os
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.distributions import OneHotCategorical, Categorical, kl_divergence
from torch.amp import GradScaler                # PyTorch ≥ 2.4 API
from tqdm.auto import tqdm

DEVICE      = torch.device("cuda")
FEATURE_DIM = 300
ACTION_DIM  = 3
SEQ_LEN     = 128

HORIZON_NAMES = ["5m", "15m", "1h", "4h", "1d", "5d", "30d"]
HORIZON_STEPS = [1, 3, 12, 48, 75, 375, 2250]
QUANTILES     = [0.05, 0.25, 0.50, 0.75, 0.95]

# Sanity-check AMP is available for this GPU
assert torch.cuda.is_bf16_supported() or True, "No BF16 — using FP16 AMP"
print(f"Training on : {torch.cuda.get_device_name(0)}")
print(f"Precision   : FP16 mixed (AMP)")
print("Imports OK")

Training on : NVIDIA GeForce RTX 4070 Laptop GPU
Precision   : FP16 mixed (AMP)
Imports OK


## Section 2: Download Real Market Data

In [4]:
# ── Cell 4: Download real market data (skips symbols already on disk) ─────────
import yfinance as yf

NSE_EQUITIES = {
    "RELIANCE": "RELIANCE.NS", "TCS": "TCS.NS", "HDFCBANK": "HDFCBANK.NS",
    "INFY": "INFY.NS", "ICICIBANK": "ICICIBANK.NS", "SBIN": "SBIN.NS",
    "BHARTIARTL": "BHARTIARTL.NS", "ITC": "ITC.NS", "KOTAKBANK": "KOTAKBANK.NS",
    "LT": "LT.NS", "AXISBANK": "AXISBANK.NS", "MARUTI": "MARUTI.NS",
    "WIPRO": "WIPRO.NS", "HCLTECH": "HCLTECH.NS", "SUNPHARMA": "SUNPHARMA.NS",
    "TATASTEEL": "TATASTEEL.NS", "ADANIENT": "ADANIENT.NS",
    "BAJFINANCE": "BAJFINANCE.NS", "ASIANPAINT": "ASIANPAINT.NS",
}
INDICES = {"NIFTY50": "^NSEI", "BANKNIFTY": "^NSEBANK"}
MACRO   = {
    "SP500": "^GSPC", "DOW": "^DJI", "NASDAQ": "^IXIC",
    "CRUDE_OIL": "CL=F", "GOLD": "GC=F", "SILVER": "SI=F",
    "USDINR": "USDINR=X", "DXY": "DX-Y.NYB", "VIX": "^VIX", "US10Y": "^TNX",
}
ALL_TICKERS = {**NSE_EQUITIES, **INDICES, **MACRO}

end   = datetime.now()
start = end - timedelta(days=730)
downloaded, skipped, failed = [], [], []

for name, ticker in tqdm(ALL_TICKERS.items(), desc="Downloading"):
    out_file = RAW_DIR / name.lower() / f"{name}_1d.parquet"
    if out_file.exists():
        skipped.append(name)
        continue
    try:
        df = yf.download(
            ticker, start=start.strftime("%Y-%m-%d"), end=end.strftime("%Y-%m-%d"),
            interval="1d", progress=False, auto_adjust=True,
        )
        if df.empty:
            failed.append(name)
            continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df.rename(columns={"Open":"open","High":"high","Low":"low","Close":"close","Volume":"volume"})
        cols = [c for c in ["open","high","low","close","volume"] if c in df.columns]
        df = df[cols].copy()
        df["timestamp"] = df.index
        df = df.reset_index(drop=True)[["timestamp"]+cols].dropna(subset=["close"])
        out_file.parent.mkdir(parents=True, exist_ok=True)
        df.to_parquet(out_file, index=False)
        downloaded.append(name)
    except Exception as e:
        failed.append(name)
        print(f"  [FAIL] {name}: {e}")

total = len(downloaded) + len(skipped)
print(f"\nNew: {len(downloaded)}  |  Already cached: {len(skipped)}  |  Failed: {len(failed)}")
print(f"Total available: {total} symbols")
assert total >= 10, f"Only {total} symbols — check internet connection"

Downloading:   0%|          | 0/31 [00:00<?, ?it/s]


New: 31  |  Already cached: 0  |  Failed: 0
Total available: 31 symbols


## Section 3: Feature Engineering & Dataset

In [10]:
# ── Cell 5: Feature engineering (300-dim vectors) ────────────────────────────
def compute_features(df, feature_dim=300):
    feat = pd.DataFrame(index=df.index)
    close  = df["close"].astype(float)
    high   = df["high"].astype(float)
    low    = df["low"].astype(float)
    volume = df["volume"].astype(float)

    feat["log_ret_1"]  = np.log(close / close.shift(1))
    feat["log_ret_5"]  = np.log(close / close.shift(5))
    feat["log_ret_15"] = np.log(close / close.shift(15))
    feat["log_ret_30"] = np.log(close / close.shift(30))

    feat["realized_vol_20"] = feat["log_ret_1"].rolling(20).std() * np.sqrt(252)
    feat["realized_vol_5"]  = feat["log_ret_1"].rolling(5).std()  * np.sqrt(252)
    hl = np.log(high / low.replace(0, np.nan))
    feat["parkinson_vol"] = hl.rolling(20).apply(
        lambda x: np.sqrt((x**2).sum() / (4 * len(x) * np.log(2))), raw=True)

    avg_vol = volume.rolling(20).mean()
    feat["rel_volume"]    = volume / avg_vol.replace(0, np.nan)
    feat["vol_momentum"]  = volume.rolling(5).mean() / avg_vol.replace(0, np.nan)
    feat["obv_norm"]      = (volume * np.sign(feat["log_ret_1"])).rolling(20).sum() / avg_vol.replace(0, np.nan)
    feat["hl_ratio"]      = (high - low) / close.replace(0, np.nan)
    feat["co_ratio"]      = (close - df["open"].astype(float)) / close.replace(0, np.nan)

    delta    = close.diff()
    gain     = delta.where(delta > 0, 0.0).rolling(14).mean()
    loss_val = (-delta.where(delta < 0, 0.0)).rolling(14).mean()
    feat["rsi_14"] = (100 - 100 / (1 + gain / loss_val.replace(0, np.nan))) / 100

    ema12 = close.ewm(span=12).mean(); ema26 = close.ewm(span=26).mean()
    macd  = ema12 - ema26
    feat["macd_hist"] = (macd - macd.ewm(span=9).mean()) / close.replace(0, np.nan)

    sma20 = close.rolling(20).mean(); std20 = close.rolling(20).std()
    feat["bb_pct_b"] = (close - sma20 + 2*std20) / (4*std20).replace(0, np.nan)

    tr = pd.concat([high-low, (high-close.shift()).abs(), (low-close.shift()).abs()], axis=1).max(axis=1)
    feat["atr_norm"] = tr.rolling(14).mean() / close.replace(0, np.nan)

    if "timestamp" in df.columns:
        ts  = pd.to_datetime(df["timestamp"])
        hr  = ts.dt.hour + ts.dt.minute / 60
        dow = ts.dt.dayofweek
        feat["hour_sin"] = np.sin(2*np.pi*hr  / 24)
        feat["hour_cos"] = np.cos(2*np.pi*hr  / 24)
        feat["dow_sin"]  = np.sin(2*np.pi*dow / 5)
        feat["dow_cos"]  = np.cos(2*np.pi*dow / 5)

    feat = feat.ffill().bfill().fillna(0).clip(-5, 5)
    mat  = feat.values.astype(np.float32)
    if mat.shape[1] < feature_dim:
        mat = np.concatenate([mat, np.zeros((len(mat), feature_dim-mat.shape[1]), np.float32)], axis=1)
    return mat[:, :feature_dim]

all_features, all_close = [], []
parquets = sorted(RAW_DIR.rglob("*.parquet"))
print(f"Loading {len(parquets)} parquet files from {RAW_DIR} ...")
for pf in parquets:
    df = pd.read_parquet(pf)
    if len(df) < 50:
        continue
    all_features.append(compute_features(df))
    all_close.append(df["close"].values.astype(np.float64))

assert len(all_features) > 0, f"No parquet files found in {RAW_DIR}\nRun Cell 4 first."
combined       = np.concatenate(all_features, axis=0)
combined_close = np.concatenate(all_close,    axis=0)
print(f"Features : {combined.shape}  |  Close prices : {combined_close.shape}")

Loading 31 parquet files from C:\Users\M.S. BAVESH REDDY\Desktop\brain\data\raw ...


KeyboardInterrupt: 

In [6]:
# ── Cell 6: Dataset for RSSM + Actor-Critic ──────────────────────────────────
class MarketSequenceDataset(Dataset):
    def __init__(self, features, close_prices, seq_len=128):
        self.features = torch.from_numpy(features).float()
        self.close    = close_prices
        self.seq_len  = seq_len

    def __len__(self):
        return max(0, len(self.features) - self.seq_len)

    def __getitem__(self, idx):
        obs      = self.features[idx : idx + self.seq_len]
        close_w  = self.close[idx : idx + self.seq_len]
        log_ret  = np.diff(np.log(np.maximum(close_w, 1e-8)))
        log_ret  = np.concatenate([[0.0], log_ret]).astype(np.float32)
        rewards  = log_ret.copy()

        abs_nz   = np.abs(log_ret[log_ret != 0])
        thresh   = float(np.percentile(abs_nz, 60)) if len(abs_nz) > 0 else 0.001
        act_idx  = np.where(log_ret > thresh, 0, np.where(log_ret < -thresh, 1, 2))
        actions  = np.eye(ACTION_DIM, dtype=np.float32)[act_idx]
        cont     = np.ones(self.seq_len, dtype=np.float32)

        return {
            "observations": obs,
            "actions":      torch.from_numpy(actions),
            "rewards":      torch.from_numpy(rewards),
            "continues":    torch.from_numpy(cont),
        }

# num_workers: 0 on Windows (fork issue), 2 on Linux/Mac
_nw = 0 if os.name == "nt" else 2

dataset = MarketSequenceDataset(combined, combined_close, seq_len=SEQ_LEN)
loader  = DataLoader(
    dataset, batch_size=WM_BATCH, shuffle=True,
    num_workers=_nw, pin_memory=True, drop_last=True,
)
print(f"Dataset : {len(dataset):,} sequences")
print(f"Batches : {len(loader):,}  (batch_size={WM_BATCH})")

Dataset : 15,268 sequences
Batches : 1,908  (batch_size=8)


In [7]:
# Cell 7 — TFT targets + windowed features
HORIZON_NAMES = ["5m", "15m", "1h", "4h", "1d", "5d", "30d"]
HORIZON_STEPS = [1, 3, 12, 48, 75, 375, 2250]

def compute_multi_horizon_targets(close_prices, flat_threshold=0.001):
    N = len(close_prices)
    log_ret_1 = np.log(close_prices[1:] / np.maximum(close_prices[:-1], 1e-8))
    returns_d, directions_d, volatility_d = {}, {}, {}
    for name, steps in zip(HORIZON_NAMES, HORIZON_STEPS):
        ret = np.full(N, np.nan)
        direction = np.full(N, np.nan)
        vol = np.full(N, np.nan)
        valid_end = N - steps
        if valid_end > 0:
            idx = np.arange(valid_end)
            ret[idx] = (close_prices[idx + steps] - close_prices[idx]) / np.maximum(close_prices[idx], 1e-8)
            direction[idx] = np.where(ret[idx] > flat_threshold, 2, np.where(ret[idx] < -flat_threshold, 0, 1))
        win = max(steps, 5)
        for i in range(win, min(N, len(log_ret_1))):
            vol[i] = np.std(log_ret_1[max(0, i - win) : i]) * np.sqrt(252 * 75)
        returns_d[name] = ret
        directions_d[name] = direction
        volatility_d[name] = vol
    return {"returns": returns_d, "directions": directions_d, "volatility": volatility_d}

# Create TFT windowed features (N, T, 300)
n_windows = len(combined) - SEQ_LEN
tft_features = np.lib.stride_tricks.sliding_window_view(combined, (SEQ_LEN, FEATURE_DIM)).squeeze(1).copy()
tft_close = combined_close[SEQ_LEN:]

split = int(n_windows * 0.8)
tft_train_feat, tft_val_feat = tft_features[:split], tft_features[split:]
tft_train_close, tft_val_close = tft_close[:split], tft_close[split:]

print(f"TFT windows: {tft_features.shape}")
print(f"Train: {tft_train_feat.shape}, Val: {tft_val_feat.shape}")

TFT windows: (15269, 128, 300)
Train: (12214, 128, 300), Val: (3055, 128, 300)


## Section 4: Train RSSM World Model (~20 min on T4)

In [8]:
# ── Cell 8: Model definitions (fully inlined — no external files needed) ──────

# ─────────────────────────── RSSM World Model ────────────────────────────────

class RSSM(nn.Module):
    def __init__(self, obs_dim=300, action_dim=3, det_size=512,
                 stoch_size=1024, num_cat=32, cat_dim=32, hidden=512):
        super().__init__()
        self.det_size   = det_size
        self.stoch_size = stoch_size
        self.num_cat    = num_cat
        self.cat_dim    = cat_dim

        self.gru_in  = nn.Linear(stoch_size + action_dim, hidden)
        self.gru     = nn.GRUCell(hidden, det_size)
        self.prior   = nn.Sequential(nn.Linear(det_size, hidden), nn.SiLU(), nn.Linear(hidden, num_cat * cat_dim))
        self.post    = nn.Sequential(nn.Linear(det_size + obs_dim, hidden), nn.SiLU(), nn.Linear(hidden, num_cat * cat_dim))

    def initial_state(self, B, device):
        return (torch.zeros(B, self.det_size, device=device),
                torch.zeros(B, self.stoch_size, device=device))

    def _det_step(self, h, z, a):
        x = F.silu(self.gru_in(torch.cat([z, a], -1)))
        return self.gru(x, h)

    def _sample(self, logits):
        logits = logits.view(-1, self.num_cat, self.cat_dim)
        dist   = OneHotCategorical(logits=logits)
        s      = dist.sample()
        s      = s + dist.probs - dist.probs.detach()       # straight-through
        return s.view(-1, self.stoch_size), logits

    def observe_step(self, h, z, a, obs):
        h   = self._det_step(h, z, a)
        z, post_l = self._sample(self.post(torch.cat([h, obs], -1)))
        _, prior_l = self._sample(self.prior(h))
        return h, z, {"prior_logits": prior_l, "posterior_logits": post_l}

    def imagine_step(self, h, z, a):
        h   = self._det_step(h, z, a)
        z, prior_l = self._sample(self.prior(h))
        return h, z, {"prior_logits": prior_l}

    def feat(self, h, z):
        return torch.cat([h, z], -1)

    @property
    def state_feature_dim(self):
        return self.det_size + self.stoch_size


def _mlp(in_d, hid, out_d, act=nn.SiLU):
    return nn.Sequential(nn.Linear(in_d, hid), act(), nn.Linear(hid, hid), act(), nn.Linear(hid, out_d))


class WorldModel(nn.Module):
    def __init__(self, obs_dim=300, action_dim=3):
        super().__init__()
        self.rssm     = RSSM(obs_dim=obs_dim, action_dim=action_dim)
        sd            = self.rssm.state_feature_dim          # 1536
        self.decoder  = _mlp(sd, 512, obs_dim)
        self.rew_head = _mlp(sd, 256, 1)
        self.cont_head= _mlp(sd, 256, 1)
        self.ret_head = nn.Sequential(nn.Linear(sd,256),nn.SiLU(),nn.Linear(256,7*5))   # 7 horizons × 5 quantiles
        self.dir_head = nn.Sequential(nn.Linear(sd,256),nn.SiLU(),nn.Linear(256,7*3))   # 7 horizons × 3 classes
        self.vol_head = nn.Sequential(nn.Linear(sd,256),nn.SiLU(),nn.Linear(256,7),nn.Softplus())

    def forward(self, obs, actions):
        B, T, _ = obs.shape
        h, z    = self.rssm.initial_state(B, obs.device)
        recons, rew_preds, cont_preds = [], [], []
        prior_ls, post_ls = [], []

        for t in range(T):
            h, z, info = self.rssm.observe_step(h, z, actions[:, t], obs[:, t])
            sf = self.rssm.feat(h, z)
            recons.append(self.decoder(sf))
            rew_preds.append(self.rew_head(sf).squeeze(-1))
            cont_preds.append(self.cont_head(sf).squeeze(-1))
            prior_ls.append(info["prior_logits"])
            post_ls.append(info["posterior_logits"])

        return {
            "recon":            torch.stack(recons,    dim=1),
            "reward_pred":      torch.stack(rew_preds, dim=1),
            "continue_pred":    torch.stack(cont_preds,dim=1),
            "prior_logits":     torch.stack(prior_ls,  dim=1),
            "posterior_logits": torch.stack(post_ls,   dim=1),
        }

    def imagine(self, h, z, policy_fn, horizon=15):
        states, rewards, continues = [], [], []
        for _ in range(horizon):
            sf = self.rssm.feat(h, z)
            states.append(sf)
            a  = policy_fn(sf)
            h, z, _ = self.rssm.imagine_step(h, z, a)
            rewards.append(self.rew_head(sf).squeeze(-1))
            continues.append(self.cont_head(sf).squeeze(-1))
        return {
            "states":    torch.stack(states,    dim=1),
            "rewards":   torch.stack(rewards,   dim=1),
            "continues": torch.stack(continues, dim=1),
        }


# ─────────────────────────── World Model Loss ────────────────────────────────

def symlog(x):
    return torch.sign(x) * torch.log1p(x.abs())

class WorldModelLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.qs = [0.05, 0.25, 0.50, 0.75, 0.95]

    def forward(self, out, obs, rewards, continues):
        # Reconstruction
        recon_loss = F.mse_loss(out["recon"], symlog(obs).detach())

        # KL (categorical)
        prior_l = out["prior_logits"]       # (B, T, C, D)
        post_l  = out["posterior_logits"]
        B, T, C, D = prior_l.shape
        prior_d = OneHotCategorical(logits=prior_l.view(B*T, C, D))
        post_d  = OneHotCategorical(logits=post_l.view(B*T, C, D))
        kl_raw  = kl_divergence(post_d, prior_d).sum(-1)           # (B*T,)
        kl_loss = kl_raw.clamp(min=1.0).mean()

        # Reward
        rew_loss  = F.mse_loss(out["reward_pred"], symlog(rewards).detach())

        # Continue
        cont_loss = F.binary_cross_entropy_with_logits(out["continue_pred"], continues)

        total = recon_loss + kl_loss + rew_loss + cont_loss
        return {"total": total, "recon": recon_loss, "kl": kl_loss,
                "reward": rew_loss, "continue": cont_loss}


# ─────────────────────────── Actor-Critic ────────────────────────────────────

class Actor(nn.Module):
    def __init__(self, state_dim, action_dim=3, hidden=256):
        super().__init__()
        self.net = _mlp(state_dim, hidden, action_dim)
        self.action_dim = action_dim

    def forward(self, sf):
        return Categorical(logits=self.net(sf))

    def get_action(self, sf, greedy=False):
        dist = self.forward(sf)
        idx = dist.probs.argmax(-1) if greedy else dist.sample()
        return F.one_hot(idx, self.action_dim).float()


class Critic(nn.Module):
    def __init__(self, state_dim, hidden=256):
        super().__init__()
        self.net = _mlp(state_dim, hidden, 1)

    def forward(self, sf):
        return self.net(sf).squeeze(-1)


class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim=3):
        super().__init__()
        self.actor         = Actor(state_dim, action_dim)
        self.critic        = Critic(state_dim)
        self.target_critic = Critic(state_dim)
        self.target_critic.load_state_dict(self.critic.state_dict())
        self.gamma  = 0.997
        self.lambda_= 0.95
        self.ent_w  = 3e-4
        self.ema    = 0.98

    def compute_returns(self, rewards, values, continues):
        B, H  = rewards.shape
        rets  = torch.zeros_like(rewards)
        rets[:,-1] = rewards[:,-1] + self.gamma * continues[:,-1] * values[:,-1]
        for t in reversed(range(H-1)):
            td = rewards[:,t] + self.gamma * continues[:,t] * values[:,t+1]
            rets[:,t] = self.lambda_ * rets[:,t+1] + (1-self.lambda_) * td
        return rets

    def actor_loss(self, states, returns):
        B, H, D = states.shape
        sf = states.reshape(B*H, D)
        r  = returns.reshape(B*H)
        dist = self.actor(sf)
        lp   = dist.log_prob(dist.sample())
        adv  = (r - r.mean()) / (r.std().clamp(1e-6))
        pol  = -(lp * adv.detach()).mean()
        ent  = dist.entropy().mean()
        return {"total": pol - self.ent_w * ent, "policy": pol, "entropy": ent}

    def critic_loss(self, states, returns):
        B, H, D = states.shape
        v = self.critic(states.reshape(B*H, D))
        return F.mse_loss(v, returns.reshape(B*H).detach())

    def update_target(self):
        for p, tp in zip(self.critic.parameters(), self.target_critic.parameters()):
            tp.data.mul_(self.ema).add_(p.data, alpha=1-self.ema)

    def get_action(self, sf, greedy=False):
        return self.actor.get_action(sf, greedy)


# ─────────────────────────── TFT ─────────────────────────────────────────────

class GatedLinearUnit(nn.Module):
    def __init__(self, in_d, out_d):
        super().__init__()
        self.fc   = nn.Linear(in_d, out_d)
        self.gate = nn.Linear(in_d, out_d)
    def forward(self, x):
        return self.fc(x) * torch.sigmoid(self.gate(x))

class GRN(nn.Module):
    def __init__(self, in_d, hid, out_d, ctx_d=None, drop=0.1):
        super().__init__()
        self.fc1  = nn.Linear(in_d, hid)
        self.fc2  = nn.Linear(hid, out_d)
        self.glu  = GatedLinearUnit(out_d, out_d)
        self.ln   = nn.LayerNorm(out_d)
        self.drop = nn.Dropout(drop)
        self.ctx  = nn.Linear(ctx_d, hid, bias=False) if ctx_d else None
        self.skip = nn.Linear(in_d, out_d) if in_d != out_d else None
    def forward(self, x, ctx=None):
        res = self.skip(x) if self.skip else x
        h   = self.fc1(x)
        if self.ctx and ctx is not None:
            h = h + self.ctx(ctx)
        h = F.elu(h)
        h = self.drop(self.fc2(h))
        return self.ln(self.glu(h) + res)

class MHA(nn.Module):
    def __init__(self, d, heads, drop=0.1):
        super().__init__()
        self.heads = heads
        self.dk    = d // heads
        self.Wq = nn.Linear(d,d); self.Wk = nn.Linear(d,d)
        self.Wv = nn.Linear(d,d); self.Wo = nn.Linear(d,d)
        self.drop = nn.Dropout(drop)
    def forward(self, q, k, v):
        B, T, _ = q.shape
        def proj(W, x, slen):
            return W(x).view(B, slen, self.heads, self.dk).transpose(1,2)
        Q = proj(self.Wq, q, T);  K = proj(self.Wk, k, k.shape[1])
        V = proj(self.Wv, v, v.shape[1])
        sc = (Q @ K.transpose(-2,-1)) / math.sqrt(self.dk)
        aw = self.drop(F.softmax(sc, -1))
        out= (aw @ V).transpose(1,2).contiguous().view(B, T, -1)
        return self.Wo(out), aw.mean(1)

class TemporalFusionTransformer(nn.Module):
    def __init__(self, temporal_dim=300, static_dim=32, hidden_dim=128,
                 num_heads=4, lstm_layers=2, dropout=0.1,
                 num_horizons=7, num_quantiles=5):
        super().__init__()
        H, Q = num_horizons, num_quantiles
        self.static_enc  = GRN(static_dim, hidden_dim, hidden_dim, drop=dropout)
        self.temp_proj   = nn.Linear(temporal_dim, hidden_dim)
        self.vsn         = nn.Sequential(nn.Linear(hidden_dim,hidden_dim),nn.ELU(),
                                          nn.Linear(hidden_dim,10),nn.Softmax(dim=-1))
        self.lstm        = nn.LSTM(hidden_dim, hidden_dim, lstm_layers, batch_first=True,
                                   dropout=dropout if lstm_layers>1 else 0)
        self.lstm_glu    = GatedLinearUnit(hidden_dim, hidden_dim)
        self.lstm_ln     = nn.LayerNorm(hidden_dim)
        self.static_enr  = GRN(hidden_dim, hidden_dim, hidden_dim, ctx_d=hidden_dim, drop=dropout)
        self.attn        = MHA(hidden_dim, num_heads, dropout)
        self.attn_glu    = GatedLinearUnit(hidden_dim, hidden_dim)
        self.attn_ln     = nn.LayerNorm(hidden_dim)
        self.ff          = GRN(hidden_dim, hidden_dim, hidden_dim, drop=dropout)
        self.ret_head    = nn.Sequential(nn.Linear(hidden_dim,hidden_dim),nn.SiLU(),nn.Linear(hidden_dim,H*Q))
        self.dir_head    = nn.Sequential(nn.Linear(hidden_dim,hidden_dim),nn.SiLU(),nn.Linear(hidden_dim,H*3))
        self.vol_head    = nn.Sequential(nn.Linear(hidden_dim,hidden_dim//2),nn.SiLU(),nn.Linear(hidden_dim//2,H),nn.Softplus())
        self.num_horizons  = H
        self.num_quantiles = Q
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, temporal, static=None):
        B, T, _ = temporal.shape
        sc = self.static_enc(static) if static is not None else torch.zeros(B, self.temp_proj.out_features, device=temporal.device)
        tp = self.temp_proj(temporal)
        vsn_w = self.vsn(tp.mean(1))
        lo, _ = self.lstm(tp)
        lg     = self.lstm_ln(self.lstm_glu(lo) + tp)
        se     = self.static_enr(lg, ctx=sc.unsqueeze(1).expand(-1,T,-1))
        ao, aw = self.attn(se, se, se)
        ae     = self.attn_ln(self.attn_glu(ao) + se)
        tf     = self.ff(ae)
        lf     = tf[:, -1]
        rq = self.ret_head(lf).view(B, self.num_horizons, self.num_quantiles)
        dl = self.dir_head(lf).view(B, self.num_horizons, 3)
        vl = self.vol_head(lf)
        return {
            "return_quantiles": rq,
            "direction_logits": dl,
            "direction_probs":  F.softmax(dl, -1),
            "volatility":       vl,
            "attention_weights":aw,
            "feature_importance":vsn_w,
            "temporal_features":lf,
        }


# ─────────────────────────── Initialise all 3 models ─────────────────────────

world_model  = WorldModel(obs_dim=FEATURE_DIM, action_dim=ACTION_DIM).to(DEVICE)
loss_fn      = WorldModelLoss()
state_dim    = world_model.rssm.state_feature_dim          # 1536

actor_critic = ActorCritic(state_dim=state_dim, action_dim=ACTION_DIM).to(DEVICE)

tft_model    = TemporalFusionTransformer(
    temporal_dim=FEATURE_DIM, static_dim=32, hidden_dim=128,
    num_heads=4, lstm_layers=2, dropout=0.1,
).to(DEVICE)

def n(m): return sum(p.numel() for p in m.parameters())
print(f"WorldModel   {n(world_model):>10,} params  |  state_dim={state_dim}")
print(f"Actor-Critic {n(actor_critic):>10,} params")
print(f"TFT          {n(tft_model):>10,} params")
print(f"Total        {n(world_model)+n(actor_critic)+n(tft_model):>10,} params")

WorldModel    7,150,957 params  |  state_dim=1536
Actor-Critic  1,379,077 params
TFT             709,321 params
Total         9,239,355 params


In [9]:
# ── Cell 9: Train RSSM World Model (FP16 AMP, ~20 min on T4) ─────────────────
NUM_WM_EPOCHS = 50
SAVE = Path(SAVE_DIR)          # Path obj for / operator throughout the notebook

wm_opt   = torch.optim.AdamW(world_model.parameters(), lr=1e-4, weight_decay=1e-5)
wm_sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(wm_opt, T_0=10, T_mult=2)
scaler   = GradScaler("cuda")

world_model.train()
best_wm = float("inf")
t0      = time.time()

for epoch in range(NUM_WM_EPOCHS):
    ep_losses = []
    pbar = tqdm(loader, desc=f"WM {epoch+1}/{NUM_WM_EPOCHS}", leave=False)

    for batch in pbar:
        obs      = batch["observations"].to(DEVICE, non_blocking=True)
        actions  = batch["actions"].to(DEVICE, non_blocking=True)
        rewards  = batch["rewards"].to(DEVICE, non_blocking=True)
        continues= batch["continues"].to(DEVICE, non_blocking=True)

        wm_opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            out    = world_model(obs, actions)
            losses = loss_fn(out, obs, rewards, continues)
            loss   = losses["total"]

        scaler.scale(loss).backward()
        scaler.unscale_(wm_opt)
        torch.nn.utils.clip_grad_norm_(world_model.parameters(), 100.0)
        scaler.step(wm_opt); scaler.update()

        ep_losses.append(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    wm_sched.step()
    avg = float(np.mean(ep_losses))
    if avg < best_wm:
        best_wm = avg
        torch.save(world_model.state_dict(), SAVE/"world_model_best.pt")
    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1:>3d}/{NUM_WM_EPOCHS} | Loss: {avg:.4f} | Best: {best_wm:.4f} | {time.time()-t0:.0f}s")
    if (epoch+1) % 10 == 0:
        torch.save(world_model.state_dict(), SAVE/f"world_model_ep{epoch+1}.pt")

torch.save(world_model.state_dict(), SAVE/"world_model.pt")
print(f"\nWorld Model done: {time.time()-t0:.0f}s | Best: {best_wm:.4f}")
torch.cuda.empty_cache()

WM 1/50:   0%|          | 0/1908 [00:00<?, ?it/s]

WM 2/50:   0%|          | 0/1908 [00:00<?, ?it/s]

WM 3/50:   0%|          | 0/1908 [00:00<?, ?it/s]

WM 4/50:   0%|          | 0/1908 [00:00<?, ?it/s]

WM 5/50:   0%|          | 0/1908 [00:00<?, ?it/s]

...

WM 32/50:   0%|          | 0/1908 [00:00<?, ?it/s]

WM 33/50:   0%|          | 0/1908 [00:00<?, ?it/s]

WM 34/50:   0%|          | 0/1908 [00:00<?, ?it/s]

WM 35/50:   0%|          | 0/1908 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [13]:
# ── COMPREHENSIVE PORTABLE SAVE — everything needed to resume on any laptop ──
import json, shutil, pickle
from pathlib import Path

WEIGHTS_DIR = Path(r"C:\Users\M.S. BAVESH REDDY\Downloads\weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

saved = []

# ═══════════════════════════════════════════════════════════════════════════════
# 1. ALL MODEL WEIGHTS (World Model — every available epoch)
# ═══════════════════════════════════════════════════════════════════════════════
SRC = Path(r"C:\Users\M.S. BAVESH REDDY\Desktop\brain\data\models")
if SRC.exists():
    for f in SRC.glob("*.pt"):
        dst = WEIGHTS_DIR / f.name
        shutil.copy2(f, dst)
        saved.append(("epoch_checkpoint", f.name, dst.stat().st_size/1e6))

# Save latest in-memory weights
torch.save(world_model.state_dict(), WEIGHTS_DIR / "world_model_latest.pt")
saved.append(("model_weights", "world_model_latest.pt", (WEIGHTS_DIR/"world_model_latest.pt").stat().st_size/1e6))

torch.save(actor_critic.state_dict(), WEIGHTS_DIR / "actor_critic.pt")
saved.append(("model_weights", "actor_critic.pt", (WEIGHTS_DIR/"actor_critic.pt").stat().st_size/1e6))

torch.save(tft_model.state_dict(), WEIGHTS_DIR / "tft_model.pt")
saved.append(("model_weights", "tft_model.pt", (WEIGHTS_DIR/"tft_model.pt").stat().st_size/1e6))

# ═══════════════════════════════════════════════════════════════════════════════
# 2. FULL TRAINING CHECKPOINT (optimizer + scheduler + scaler + epoch)
# ═══════════════════════════════════════════════════════════════════════════════
full_ckpt = {
    "epoch": epoch + 1,
    "best_loss": best_wm,
    "world_model_state_dict": world_model.state_dict(),
    "optimizer_state_dict": wm_opt.state_dict(),
    "scheduler_state_dict": wm_sched.state_dict(),
    "scaler_state_dict": scaler.state_dict(),
    "config": {
        "NUM_WM_EPOCHS": NUM_WM_EPOCHS,
        "WM_BATCH": WM_BATCH,
        "TFT_BATCH": TFT_BATCH,
        "SEQ_LEN": SEQ_LEN,
        "FEATURE_DIM": FEATURE_DIM,
        "ACTION_DIM": ACTION_DIM,
        "HORIZON_NAMES": HORIZON_NAMES,
        "HORIZON_STEPS": HORIZON_STEPS,
        "QUANTILES": QUANTILES,
    },
}
torch.save(full_ckpt, WEIGHTS_DIR / "world_model_full_checkpoint.pt")
saved.append(("full_checkpoint", "world_model_full_checkpoint.pt", (WEIGHTS_DIR/"world_model_full_checkpoint.pt").stat().st_size/1e6))

# ═══════════════════════════════════════════════════════════════════════════════
# 3. RAW MARKET DATA (31 parquet files — so new laptop skips download)
# ═══════════════════════════════════════════════════════════════════════════════
DATA_DST = WEIGHTS_DIR / "data" / "raw"
DATA_DST.mkdir(parents=True, exist_ok=True)
raw_src = Path(r"C:\Users\M.S. BAVESH REDDY\Desktop\brain\data\raw")
data_count = 0
for pf in raw_src.rglob("*.parquet"):
    rel = pf.relative_to(raw_src)
    dst = DATA_DST / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(pf, dst)
    data_count += 1
saved.append(("market_data", f"{data_count} parquet files", sum(f.stat().st_size for f in DATA_DST.rglob("*.parquet"))/1e6))

# ═══════════════════════════════════════════════════════════════════════════════
# 4. PROCESSED FEATURES + CLOSE PRICES (skip re-computation on new laptop)
# ═══════════════════════════════════════════════════════════════════════════════
np.savez_compressed(
    WEIGHTS_DIR / "processed_features.npz",
    combined=combined,
    combined_close=combined_close,
)
saved.append(("processed_data", "processed_features.npz", (WEIGHTS_DIR/"processed_features.npz").stat().st_size/1e6))

# Also save TFT split data
np.savez_compressed(
    WEIGHTS_DIR / "tft_split_data.npz",
    tft_train_feat=tft_train_feat,
    tft_val_feat=tft_val_feat,
    tft_train_close=tft_train_close,
    tft_val_close=tft_val_close,
)
saved.append(("processed_data", "tft_split_data.npz", (WEIGHTS_DIR/"tft_split_data.npz").stat().st_size/1e6))

# ═══════════════════════════════════════════════════════════════════════════════
# 5. TRAINING HISTORY (full per-epoch loss log)
# ═══════════════════════════════════════════════════════════════════════════════
history = {
    "model": "world_model",
    "completed_epochs": epoch + 1,
    "total_epochs": NUM_WM_EPOCHS,
    "best_loss": best_wm,
    "last_epoch_avg_loss": float(np.mean(ep_losses)),
    "last_epoch_losses_sample": [float(x) for x in ep_losses[:10]],
    "batch_size": WM_BATCH,
    "learning_rate": wm_opt.param_groups[0]["lr"],
    "training_status": "interrupted" if (epoch+1) < NUM_WM_EPOCHS else "completed",
    "remaining_epochs": NUM_WM_EPOCHS - (epoch + 1),
}
with open(WEIGHTS_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)
saved.append(("metadata", "training_history.json", 0))

# ═══════════════════════════════════════════════════════════════════════════════
# 6. ARCHITECTURE INFO
# ═══════════════════════════════════════════════════════════════════════════════
arch = {
    "world_model_params": sum(p.numel() for p in world_model.parameters()),
    "actor_critic_params": sum(p.numel() for p in actor_critic.parameters()),
    "tft_params": sum(p.numel() for p in tft_model.parameters()),
    "total_params": sum(p.numel() for p in world_model.parameters()) + sum(p.numel() for p in actor_critic.parameters()) + sum(p.numel() for p in tft_model.parameters()),
    "rssm_det_size": 512,
    "rssm_stoch_size": 1024,
    "rssm_num_cat": 32,
    "rssm_cat_dim": 32,
    "state_feature_dim": 1536,
    "tft_hidden_dim": 128,
    "tft_num_heads": 4,
    "tft_lstm_layers": 2,
    "pytorch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "gpu_name": torch.cuda.get_device_name(0),
    "python_version": sys.version,
}
with open(WEIGHTS_DIR / "architecture_info.json", "w") as f:
    json.dump(arch, f, indent=2)
saved.append(("metadata", "architecture_info.json", 0))

# ═══════════════════════════════════════════════════════════════════════════════
# 7. NOTEBOOK COPY (the actual training code)
# ═══════════════════════════════════════════════════════════════════════════════
nb_src = Path(r"c:\Users\M.S. BAVESH REDDY\Downloads\train_brain.ipynb")
if nb_src.exists():
    shutil.copy2(nb_src, WEIGHTS_DIR / "train_brain.ipynb")
    saved.append(("notebook", "train_brain.ipynb", (WEIGHTS_DIR/"train_brain.ipynb").stat().st_size/1e6))

# ═══════════════════════════════════════════════════════════════════════════════
# 8. REQUIREMENTS.TXT
# ═══════════════════════════════════════════════════════════════════════════════
reqs = """# Brain Training Requirements
torch>=2.4.0
torchvision
numpy
pandas
pyarrow
yfinance
tqdm
lightgbm
"""
with open(WEIGHTS_DIR / "requirements.txt", "w") as f:
    f.write(reqs)
saved.append(("metadata", "requirements.txt", 0))

# ═══════════════════════════════════════════════════════════════════════════════
# 9. RESUME INSTRUCTIONS
# ═══════════════════════════════════════════════════════════════════════════════
readme = f"""# Brain Training — Resume Instructions
# =====================================
# Last trained on: {torch.cuda.get_device_name(0)}
# PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}
# Stopped at: Epoch {epoch+1}/{NUM_WM_EPOCHS} | Best Loss: {best_wm:.6f}
# Remaining: {NUM_WM_EPOCHS - (epoch+1)} World Model epochs + TFT (50) + Actor-Critic (30)
#
# FOLDER CONTENTS:
#   world_model_full_checkpoint.pt  — RESUME from here (has optimizer+scheduler+scaler)
#   world_model_latest.pt           — Latest model weights only
#   world_model_best.pt             — Best model weights (lowest loss)
#   world_model_ep10/20/30.pt       — Per-epoch snapshots
#   actor_critic.pt                 — Actor-Critic weights (not yet trained)
#   tft_model.pt                    — TFT weights (not yet trained)
#   processed_features.npz          — Pre-computed 300-dim features (skip Cell 4+5)
#   tft_split_data.npz              — TFT train/val split (skip Cell 7)
#   data/raw/*.parquet              — Raw market data (31 symbols, 2yr daily)
#   train_brain.ipynb               — Complete notebook
#   training_history.json           — Loss metrics
#   architecture_info.json          — Model config + param counts
#   requirements.txt                — pip dependencies
#
# TO RESUME ON A NEW LAPTOP:
# ──────────────────────────
# 1. Copy this entire "weights" folder to the new machine
# 2. pip install -r requirements.txt
# 3. pip install torch --index-url https://download.pytorch.org/whl/cu126
# 4. Open train_brain.ipynb
# 5. Run Cells 1-3 (setup)
# 6. In Cell 2, set BASE_DIR to point to this weights/data/ folder
# 7. Skip Cell 4 (data already in data/raw/)
# 8. Run Cell 5-8 (features + models)
#    OR load processed_features.npz to skip Cell 5:
#      data = np.load("processed_features.npz")
#      combined = data["combined"]
#      combined_close = data["combined_close"]
# 9. Before Cell 9, load checkpoint:
#      ckpt = torch.load("world_model_full_checkpoint.pt")
#      world_model.load_state_dict(ckpt["world_model_state_dict"])
#      wm_opt.load_state_dict(ckpt["optimizer_state_dict"])
#      wm_sched.load_state_dict(ckpt["scheduler_state_dict"])
#      scaler.load_state_dict(ckpt["scaler_state_dict"])
#      start_epoch = ckpt["epoch"]  # = {epoch+1}
#      best_wm = ckpt["best_loss"]  # = {best_wm:.6f}
# 10. Change Cell 9 loop: for epoch in range(start_epoch, NUM_WM_EPOCHS):
# 11. Run Cell 9 to finish remaining {NUM_WM_EPOCHS-(epoch+1)} epochs
# 12. Run Cells 10-13 for TFT + Actor-Critic + Validation
"""
with open(WEIGHTS_DIR / "RESUME_INSTRUCTIONS.txt", "w", encoding="utf-8") as f:
    f.write(readme)
saved.append(("metadata", "RESUME_INSTRUCTIONS.txt", 0))

# ═══════════════════════════════════════════════════════════════════════════════
# FINAL REPORT
# ═══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("  COMPLETE PORTABLE SAVE — READY TO MOVE TO ANY LAPTOP")
print("=" * 70)

categories = {}
for cat, name, mb in saved:
    if cat not in categories:
        categories[cat] = []
    categories[cat].append((name, mb))

for cat, items in categories.items():
    print(f"\n  [{cat.upper()}]")
    for name, mb in items:
        if mb > 0:
            print(f"    {name:<45s}  {mb:>7.1f} MB")
        else:
            print(f"    {name}")

# Total size
total_bytes = sum(f.stat().st_size for f in WEIGHTS_DIR.rglob("*") if f.is_file())
total_files = len(list(WEIGHTS_DIR.rglob("*")))
print(f"\n{'='*70}")
print(f"  TOTAL: {total_bytes/1e6:.1f} MB  |  {total_files} files")
print(f"  PATH:  {WEIGHTS_DIR}")
print(f"  STATUS: Epoch {epoch+1}/{NUM_WM_EPOCHS} | Best Loss: {best_wm:.6f}")
print(f"  REMAINING: WM {NUM_WM_EPOCHS-(epoch+1)} ep + TFT 50 ep + AC 30 ep")
print(f"{'='*70}")

  COMPLETE PORTABLE SAVE — READY TO MOVE TO ANY LAPTOP

  [EPOCH_CHECKPOINT]
    world_model_best.pt                               28.6 MB
    world_model_ep10.pt                               28.6 MB
    world_model_ep20.pt                               28.6 MB
    world_model_ep30.pt                               28.6 MB

  [MODEL_WEIGHTS]
    world_model_latest.pt                             28.6 MB
    actor_critic.pt                                    5.5 MB
    tft_model.pt                                       2.9 MB

  [FULL_CHECKPOINT]
    world_model_full_checkpoint.pt                    76.3 MB

  [MARKET_DATA]
    31 parquet files                                   0.8 MB

  [PROCESSED_DATA]
    processed_features.npz                             1.0 MB
    tft_split_data.npz                               123.1 MB

  [METADATA]
    training_history.json
    architecture_info.json
    requirements.txt
    RESUME_INSTRUCTIONS.txt

  [NOTEBOOK]
    train_brain.ipynb             

In [ ]:
# ── Cell 9b: SAVE ALL World-Model training artifacts ──────────────────────────
# Destination: C:\Users\M.S. BAVESH REDDY\Downloads\weights
# Saves: every epoch checkpoint, optimizer, scheduler, scaler, loss log, config
import shutil, json, os
from pathlib import Path

WEIGHTS_DIR = Path(r"C:\Users\M.S. BAVESH REDDY\Downloads\weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

SRC_DIR = Path(r"C:\Users\M.S. BAVESH REDDY\Desktop\brain\data\models")

# ── 1. Copy all existing .pt checkpoint files from data/models/ ──────────────
copied = []
for pt_file in sorted(SRC_DIR.glob("world_model*.pt")):
    dst = WEIGHTS_DIR / pt_file.name
    shutil.copy2(pt_file, dst)
    copied.append(pt_file.name)
    print(f"  [COPIED] {pt_file.name}  →  {dst}  ({pt_file.stat().st_size/1e6:.1f} MB)")

# ── 2. Save current in-memory model weights (latest state, may be newer) ─────
try:
    torch.save(world_model.state_dict(), WEIGHTS_DIR / "world_model_latest_inmemory.pt")
    print(f"  [SAVED]  world_model_latest_inmemory.pt  (current GPU state)")
except Exception as e:
    print(f"  [SKIP]   world_model_latest_inmemory.pt — not in memory: {e}")

# ── 3. Save optimizer state dict ─────────────────────────────────────────────
try:
    torch.save(wm_opt.state_dict(), WEIGHTS_DIR / "world_model_optimizer.pt")
    print(f"  [SAVED]  world_model_optimizer.pt")
except Exception as e:
    print(f"  [SKIP]   optimizer — not in memory: {e}")

# ── 4. Save scheduler state dict ─────────────────────────────────────────────
try:
    torch.save(wm_sched.state_dict(), WEIGHTS_DIR / "world_model_scheduler.pt")
    print(f"  [SAVED]  world_model_scheduler.pt")
except Exception as e:
    print(f"  [SKIP]   scheduler — not in memory: {e}")

# ── 5. Save GradScaler state dict ────────────────────────────────────────────
try:
    torch.save(scaler.state_dict(), WEIGHTS_DIR / "world_model_scaler.pt")
    print(f"  [SAVED]  world_model_scaler.pt")
except Exception as e:
    print(f"  [SKIP]   scaler — not in memory: {e}")

# ── 6. Save per-epoch loss history (if available) ────────────────────────────
try:
    # ep_losses was the last epoch's batch losses; reconstruct what we can
    loss_log = {
        "best_loss": best_wm,
        "note": "Training was interrupted — these are the values at interruption time"
    }
    with open(WEIGHTS_DIR / "world_model_training_log.json", "w") as f:
        json.dump(loss_log, f, indent=2)
    print(f"  [SAVED]  world_model_training_log.json  (best_loss={best_wm:.6f})")
except Exception as e:
    print(f"  [SKIP]   training log — variable not found: {e}")

# ── 7. Save full training config / hyperparameters ───────────────────────────
config = {
    "model": "RSSM WorldModel",
    "total_params": sum(p.numel() for p in world_model.parameters()),
    "obs_dim": FEATURE_DIM,
    "action_dim": ACTION_DIM,
    "seq_len": SEQ_LEN,
    "batch_size": WM_BATCH,
    "planned_epochs": 50,
    "lr": 1e-4,
    "weight_decay": 1e-5,
    "scheduler": "CosineAnnealingWarmRestarts(T_0=10, T_mult=2)",
    "amp_dtype": "float16",
    "grad_clip": 100.0,
    "rssm_det_size": 512,
    "rssm_stoch_size": 1024,
    "rssm_num_cat": 32,
    "rssm_cat_dim": 32,
    "rssm_hidden": 512,
    "gpu": torch.cuda.get_device_name(0),
    "vram_gb": round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1) if hasattr(torch.cuda.get_device_properties(0), 'total_mem') else "N/A",
    "pytorch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
}
try:
    config["vram_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
except:
    pass

with open(WEIGHTS_DIR / "world_model_config.json", "w") as f:
    json.dump(config, f, indent=2)
print(f"  [SAVED]  world_model_config.json")

# ── 8. Save model architecture summary as text ───────────────────────────────
with open(WEIGHTS_DIR / "world_model_architecture.txt", "w") as f:
    f.write(f"WorldModel — {sum(p.numel() for p in world_model.parameters()):,} parameters\n")
    f.write("=" * 70 + "\n\n")
    f.write(str(world_model))
    f.write("\n\n" + "=" * 70 + "\n")
    f.write("Parameter shapes:\n")
    for name, param in world_model.named_parameters():
        f.write(f"  {name:50s}  {str(list(param.shape)):20s}  ({param.numel():>10,})\n")
print(f"  [SAVED]  world_model_architecture.txt")

# ── 9. Save individual layer weights as separate files ────────────────────────
layers_dir = WEIGHTS_DIR / "layer_weights"
layers_dir.mkdir(exist_ok=True)
for name, param in world_model.named_parameters():
    safe_name = name.replace(".", "_") + ".pt"
    torch.save(param.data.cpu(), layers_dir / safe_name)
print(f"  [SAVED]  {len(list(world_model.named_parameters()))} individual layer weight files → layer_weights/")

# ── Summary ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"  ALL ARTIFACTS SAVED TO: {WEIGHTS_DIR}")
print("=" * 60)
total_files = list(WEIGHTS_DIR.rglob("*"))
total_files = [f for f in total_files if f.is_file()]
total_size = sum(f.stat().st_size for f in total_files)
print(f"  Total files : {len(total_files)}")
print(f"  Total size  : {total_size/1e6:.1f} MB")
print()
for f in sorted(WEIGHTS_DIR.iterdir()):
    if f.is_file():
        print(f"    {f.name:45s} {f.stat().st_size/1e6:8.1f} MB")
    elif f.is_dir():
        sub_count = len(list(f.rglob("*")))
        sub_size = sum(x.stat().st_size for x in f.rglob("*") if x.is_file())
        print(f"    {f.name + '/':45s} {sub_size/1e6:8.1f} MB  ({sub_count} files)")

## Section 5: Train TFT (~15 min on T4)

In [ ]:
# ── Cell 10: Train TFT ────────────────────────────────────────────────────────
NUM_TFT_EPOCHS = 50
SAVE = SAVE_DIR

tft_opt   = torch.optim.AdamW(tft_model.parameters(), lr=1e-3, weight_decay=1e-5)
tft_sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(tft_opt, T_0=50, T_mult=2)
tft_scaler= GradScaler("cuda")
best_tft  = float("inf")
t0        = time.time()

def _tft_targets(close_prices, flat=0.001):
    N   = len(close_prices)
    lr1 = np.log(np.maximum(close_prices[1:], 1e-8) / np.maximum(close_prices[:-1], 1e-8))
    ret_d, dir_d, vol_d = {}, {}, {}
    for name, steps in zip(HORIZON_NAMES, HORIZON_STEPS):
        ret = np.full(N, np.nan, dtype=np.float32)
        d   = np.full(N, np.nan, dtype=np.float32)
        end = N - steps
        if end > 0:
            idx      = np.arange(end)
            ret[idx] = (close_prices[idx+steps] - close_prices[idx]) / np.maximum(close_prices[idx], 1e-8)
            d[idx]   = np.where(ret[idx]>flat, 2, np.where(ret[idx]<-flat, 0, 1))
        win = max(steps, 5)
        vol = np.full(N, 0.0, dtype=np.float32)
        for i in range(win, min(N, len(lr1))):
            vol[i] = float(np.std(lr1[max(0, i-win):i])) * math.sqrt(252*75)
        ret_d[name]=ret; dir_d[name]=d; vol_d[name]=vol
    return {"returns":ret_d, "directions":dir_d, "volatility":vol_d}

targets   = _tft_targets(tft_train_close)
N_train   = tft_train_feat.shape[0]
valid     = np.ones(N_train, bool)
for nm in HORIZON_NAMES:
    valid &= ~np.isnan(targets["returns"][nm])
    valid &= ~np.isnan(targets["directions"][nm])
valid_idx = np.where(valid)[0]
print(f"TFT valid samples : {len(valid_idx)}/{N_train}  |  batch_size={TFT_BATCH}")

tft_model.train()
for epoch in range(NUM_TFT_EPOCHS):
    np.random.shuffle(valid_idx)
    ep_losses = []
    for i in range(0, len(valid_idx)-TFT_BATCH, TFT_BATCH):
        idx    = valid_idx[i:i+TFT_BATCH]
        temp_b = torch.tensor(tft_train_feat[idx], dtype=torch.float32, device=DEVICE)
        ret_b  = torch.tensor(np.stack([targets["returns"][h][idx]    for h in HORIZON_NAMES],1), dtype=torch.float32, device=DEVICE)
        dir_b  = torch.tensor(np.stack([targets["directions"][h][idx] for h in HORIZON_NAMES],1), dtype=torch.long,    device=DEVICE)
        vol_b  = torch.tensor(np.nan_to_num(np.stack([targets["volatility"][h][idx] for h in HORIZON_NAMES],1),0), dtype=torch.float32, device=DEVICE)

        tft_opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            out  = tft_model(temp_b)
            errs = ret_b.unsqueeze(-1) - out["return_quantiles"]
            qs   = torch.tensor(QUANTILES, device=DEVICE).view(1,1,-1)
            ql   = torch.where(errs>=0, qs*errs, (qs-1)*errs).mean()
            dl   = F.cross_entropy(out["direction_logits"].reshape(-1,3), dir_b.reshape(-1))
            vl   = F.mse_loss(out["volatility"], vol_b)
            loss = ql*2.0 + dl*1.0 + vl*0.5
        tft_scaler.scale(loss).backward()
        tft_scaler.unscale_(tft_opt)
        torch.nn.utils.clip_grad_norm_(tft_model.parameters(), 1.0)
        tft_scaler.step(tft_opt); tft_scaler.update()
        ep_losses.append(loss.item())
    tft_sched.step()

    avg = float(np.mean(ep_losses)) if ep_losses else 0.0
    if avg < best_tft:
        best_tft = avg
        torch.save(tft_model.state_dict(), SAVE/"tft_model_best.pt")
    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1:>3d}/{NUM_TFT_EPOCHS} | Loss: {avg:.4f} | Best: {best_tft:.4f} | {time.time()-t0:.0f}s")
    if (epoch+1) % 10 == 0:
        torch.save(tft_model.state_dict(), SAVE/f"tft_ep{epoch+1}.pt")

torch.save(tft_model.state_dict(), SAVE/"tft_model.pt")
print(f"\nTFT done: {time.time()-t0:.0f}s | Best: {best_tft:.4f}")
torch.cuda.empty_cache()

## Section 6: Train Actor-Critic (~10 min on T4)

In [ ]:
# ── Cell 11: Train Actor-Critic on imagined trajectories (FP16 AMP, ~10 min) ──
IMAGINATION_EPOCHS = 30
HORIZON = 15

ac_opt_actor  = torch.optim.AdamW(actor_critic.actor.parameters(),  lr=3e-5)
ac_opt_critic = torch.optim.AdamW(actor_critic.critic.parameters(), lr=3e-5)
ac_scaler_a   = GradScaler("cuda")
ac_scaler_c   = GradScaler("cuda")

world_model.eval()
t0 = time.time()

for epoch in range(IMAGINATION_EPOCHS):
    a_losses, c_losses = [], []
    pbar = tqdm(loader, desc=f"AC {epoch+1}/{IMAGINATION_EPOCHS}", leave=False)

    for batch in pbar:
        obs     = batch["observations"].to(DEVICE, non_blocking=True)
        actions = batch["actions"].to(DEVICE, non_blocking=True)
        B       = obs.shape[0]

        # Encode to get starting latent state
        t_start = np.random.randint(0, obs.shape[1] - 1)
        h, z = world_model.rssm.initial_state(B, DEVICE)
        with torch.no_grad():
            for t in range(t_start):
                h, z, _ = world_model.rssm.observe_step(h, z, actions[:, t], obs[:, t])

        # Imagine H steps
        with torch.no_grad():
            imag = world_model.imagine(h, z, lambda sf: actor_critic.get_action(sf), horizon=HORIZON)

        states   = imag["states"]
        rewards  = imag["rewards"]
        continues= torch.sigmoid(imag["continues"])

        with torch.no_grad():
            vals    = actor_critic.target_critic(states.reshape(-1, states.shape[-1])).reshape(B, HORIZON)
            returns = actor_critic.compute_returns(rewards, vals, continues)

        # Critic step
        ac_opt_critic.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            c_loss = actor_critic.critic_loss(states, returns)
        ac_scaler_c.scale(c_loss).backward()
        ac_scaler_c.step(ac_opt_critic); ac_scaler_c.update()

        # Actor step
        ac_opt_actor.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            a_out = actor_critic.actor_loss(states.detach(), returns.detach())
        ac_scaler_a.scale(a_out["total"]).backward()
        ac_scaler_a.step(ac_opt_actor); ac_scaler_a.update()

        actor_critic.update_target()
        a_losses.append(a_out["total"].item())
        c_losses.append(c_loss.item())
        pbar.set_postfix(a=f"{a_out['total'].item():.4f}", c=f"{c_loss.item():.4f}")

    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1:>3d}/{IMAGINATION_EPOCHS} | Actor: {np.mean(a_losses):.4f} | Critic: {np.mean(c_losses):.4f} | {time.time()-t0:.0f}s")
    if (epoch+1) % 10 == 0:
        torch.save(actor_critic.state_dict(), SAVE/f"actor_critic_ep{epoch+1}.pt")

torch.save(actor_critic.state_dict(), SAVE/"actor_critic.pt")
print(f"\nActor-Critic done: {time.time()-t0:.0f}s")
torch.cuda.empty_cache()

## Section 7: Validation

In [ ]:
# ── Cell 12: Validate trained models ─────────────────────────────────────────
print("="*60)
print("  Validation")
print("="*60)

wm_v  = WorldModel(obs_dim=FEATURE_DIM, action_dim=ACTION_DIM).to(DEVICE)
wm_v.load_state_dict(torch.load(SAVE/"world_model.pt", map_location=DEVICE, weights_only=True))
wm_v.eval()
print(f"[OK] WorldModel   {sum(p.numel() for p in wm_v.parameters()):,} params")

tft_v = TemporalFusionTransformer(temporal_dim=FEATURE_DIM).to(DEVICE)
tft_v.load_state_dict(torch.load(SAVE/"tft_model.pt", map_location=DEVICE, weights_only=True))
tft_v.eval()
print(f"[OK] TFT          {sum(p.numel() for p in tft_v.parameters()):,} params")

ac_v  = ActorCritic(state_dim=state_dim, action_dim=ACTION_DIM).to(DEVICE)
ac_v.load_state_dict(torch.load(SAVE/"actor_critic.pt", map_location=DEVICE, weights_only=True))
ac_v.eval()
print(f"[OK] Actor-Critic {sum(p.numel() for p in ac_v.parameters()):,} params")

# TFT direction accuracy
print("\n-- TFT Direction Accuracy --")
val_tgts = _tft_targets(tft_val_close)
nv = min(200, len(tft_val_feat))
with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
    vb    = torch.tensor(tft_val_feat[:nv], dtype=torch.float32, device=DEVICE)
    vout  = tft_v(vb)
dprobs = vout["direction_probs"].float().cpu().numpy()
for h_i, h_nm in enumerate(HORIZON_NAMES):
    y  = val_tgts["directions"][h_nm][:nv]
    ok = ~np.isnan(y)
    if ok.sum() > 0:
        acc = (dprobs[ok, h_i].argmax(1) == y[ok]).mean() * 100
        print(f"  {h_nm:>4s}: {acc:.1f}%")

# RSSM reconstruction
print("\n-- RSSM Reconstruction --")
tb = next(iter(loader))
with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
    to_ = tb["observations"][:4].to(DEVICE)
    ta_ = tb["actions"][:4].to(DEVICE)
    wo  = wm_v(to_, ta_)
    mse = F.mse_loss(wo["recon"].float(), to_.float()).item()
print(f"  Recon shape: {wo['recon'].shape}  |  MSE: {mse:.4f}")

# AC imagination smoke test
print("\n-- Actor-Critic Imagination --")
with torch.no_grad():
    h0, z0 = wm_v.rssm.initial_state(4, DEVICE)
    im = wm_v.imagine(h0, z0, lambda sf: ac_v.get_action(sf, greedy=True), horizon=HORIZON)
    print(f"  Imagined states: {im['states'].shape}")
    print(f"  Rewards mean:    {im['rewards'].mean().item():.4f}")

print("\nAll models validated!")
del wm_v, tft_v, ac_v
torch.cuda.empty_cache()

## Section 8: Model Summary

In [ ]:
# ── Cell 13: Summarise saved models ───────────────────────────────────────────
SAVE = Path(SAVE_DIR)
model_files = ["world_model.pt", "tft_model.pt", "actor_critic.pt"]

print(f"Models saved to: {SAVE}\n")
total_mb = 0
for f in model_files:
    src = SAVE / f
    if src.exists():
        sz = src.stat().st_size / 1e6
        total_mb += sz
        print(f"  [OK] {f}  ({sz:.1f} MB)")
    else:
        print(f"  [WARN] {f} not found")

print(f"\nTotal: {total_mb:.1f} MB")
print()
print("Validate with:")
print("  python scripts/validate_models.py")
print()
print("To zip for backup:")
print(f"  cd '{SAVE.parent.parent}' && zip -r brain_models.zip data/models/")